# Query for the data

In [1]:
import pystac

collection = pystac.Collection.from_file(
    "https://capella-open-data.s3.us-west-2.amazonaws.com/stac/capella-open-data-ieee-data-contest/collection.json"
)

# Check first item to see available properties
for i, item in enumerate(collection.get_items()):
    if i == 0:
        print("Available properties:")
        for key, value in item.properties.items():
            print(f"  {key}: {value}")
        print("\nAvailable assets:")
        for key in item.assets.keys():
            print(f"  {key}")
        break

Available properties:
  proj:centroid: {'lon': -155.286900345, 'lat': 19.421037849999998}
  proj:shape: [46671, 51171]
  proj:transform: [0.25780129027064413, 0, 253270.1187375269, 0, -0.25780129027064413, 2155025.709549977, 0, 0, 1]
  datetime: 2025-11-12T02:24:47.676937Z
  start_datetime: 2025-11-12T02:24:41.730047Z
  end_datetime: 2025-11-12T02:24:53.623828Z
  locale:datetime: 2025-11-11T16:24:47.676937-1000
  locale:timezone: Pacific/Honolulu
  locale:time: 16:24:47.676937
  platform: capella-13
  constellation: capella
  instruments: ['capella-radar-13']
  sar:instrument_mode: spotlight
  sar:frequency_band: X
  sar:center_frequency: 9.6
  sar:polarizations: ['HH']
  sat:orbit_state: ascending
  sar:product_type: GEO
  sar:pixel_spacing_range: 0.26
  sar:pixel_spacing_azimuth: 0.26
  sar:looks_range: 1
  sar:looks_azimuth: 3
  sar:looks_equivalent_number: 3
  sar:resolution_range: 0.32
  capella:resolution_ground_range: 0.4
  sar:resolution_azimuth: 0.56
  view:incidence_angle: 52

In [2]:
import pystac
from multiprocessing import Pool, cpu_count
from tqdm import tqdm

def process_item(item_link):
    """Process a single item and extract key parameters"""
    try:
        item = pystac.Item.from_file(item_link)
        props = item.properties
        
        # Get centroid from proj:centroid if available
        centroid = props.get('proj:centroid', {})
        
        return {
            'stac_id': item.id,
            'collect_id': props.get('capella:collect_id', ''),
            'datetime': props.get('datetime', ''),
            'start_datetime': props.get('start_datetime', ''),
            'end_datetime': props.get('end_datetime', ''),
            'center_lon': centroid.get('lon', ''),
            'center_lat': centroid.get('lat', ''),
            'platform': props.get('platform', ''),
            'constellation': props.get('constellation', ''),
            'instrument_mode': props.get('sar:instrument_mode', ''),
            'frequency_band': props.get('sar:frequency_band', ''),
            'center_frequency': props.get('sar:center_frequency', ''),
            'polarizations': ','.join(props.get('sar:polarizations', [])),
            'orbit_state': props.get('sat:orbit_state', ''),
            'product_type': props.get('sar:product_type', ''),
            'observation_direction': props.get('sar:observation_direction', ''),
            'incidence_angle': props.get('view:incidence_angle', ''),
            'azimuth': props.get('view:azimuth', ''),
            'squint_angle': props.get('capella:squint_angle', ''),
            'layover_angle': props.get('capella:layover_angle', ''),
            'look_angle': props.get('capella:look_angle', ''),
            'resolution_range': props.get('sar:resolution_range', ''),
            'resolution_azimuth': props.get('sar:resolution_azimuth', ''),
            'resolution_ground_range': props.get('capella:resolution_ground_range', ''),
            'pixel_spacing_range': props.get('sar:pixel_spacing_range', ''),
            'pixel_spacing_azimuth': props.get('sar:pixel_spacing_azimuth', ''),
            'image_length': props.get('capella:image_length', ''),
            'image_width': props.get('capella:image_width', ''),
            'looks_range': props.get('sar:looks_range', ''),
            'looks_azimuth': props.get('sar:looks_azimuth', ''),
            'orbital_plane': props.get('capella:orbital_plane', ''),
            'collection_type': props.get('capella:collection_type', ''),
        }
    except Exception as e:
        print(f"Error processing {item_link}: {e}")
        return None



In [3]:
from urllib.parse import urljoin

# Open collection
print("Opening collection...")
collection_url = "https://capella-open-data.s3.us-west-2.amazonaws.com/stac/capella-open-data-ieee-data-contest/collection.json"
collection = pystac.Collection.from_file(collection_url)

# Get item links and resolve relative URLs
print("Collecting item links...")
item_links = []
for link in collection.links:
    if link.rel == 'item':
        # Resolve relative path to absolute URL
        if link.href.startswith('./'):
            # Remove './' and join with base URL
            absolute_url = urljoin(collection_url, link.href[2:])
            item_links.append(absolute_url)
        elif link.href.startswith('http'):
            # Already absolute
            item_links.append(link.href)
        else:
            # Relative without './'
            absolute_url = urljoin(collection_url, link.href)
            item_links.append(absolute_url)

print(f"Found {len(item_links)} items")
print(f"Example URL: {item_links[0]}")  # Check if it looks right


Opening collection...
Found 1582 items
Example URL: https://capella-open-data.s3.us-west-2.amazonaws.com/stac/capella-open-data-by-datetime/capella-open-data-2025/capella-open-data-2025-11/capella-open-data-2025-11-12/CAPELLA_C13_SP_GEO_HH_20251112022441_20251112022453/CAPELLA_C13_SP_GEO_HH_20251112022441_20251112022453.json


In [4]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# Process with threading
print(f"Processing with 20 threads...")
results = []

with ThreadPoolExecutor(max_workers=20) as executor:
    futures = [executor.submit(process_item, link) for link in item_links]
    
    for future in tqdm(as_completed(futures), total=len(futures)):
        result = future.result()
        if result:
            results.append(result)

Processing with 20 threads...


100%|██████████| 1582/1582 [00:57<00:00, 27.73it/s]


In [5]:
print(f"Successfully processed {len(results)} items")

Successfully processed 1582 items


In [6]:
# Write to CSV
import csv

filename = "Capella_IEEE_DataContest_2026.csv"
if results:
    with open(filename, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)
    print(f"Saved to {filename}")

Saved to Capella_IEEE_DataContest_2026.csv


# Query over spatio-temporal region

In [7]:
import pandas as pd
from datetime import datetime, timezone

# Load the CSV
df = pd.read_csv(filename)

# Convert datetime columns to pandas datetime
df['datetime'] = pd.to_datetime(df['datetime'])
df['start_datetime'] = pd.to_datetime(df['start_datetime'])
df['end_datetime'] = pd.to_datetime(df['end_datetime'])

# Define time range
start_time = datetime(2020, 1, 1, tzinfo=timezone.utc)
end_time = datetime.now(timezone.utc)

# Define Los Angeles bounding box (approximate)
# LA is roughly at 34.05°N, -118.24°W
la_bbox = {
    'west': -118.7,
    'east': -117.7,
    'south': 33.7,
    'north': 34.4
}

# Filter by time and location
filtered_df = df[
    (df['start_datetime'] >= start_time) &
    (df['start_datetime'] <= end_time) &
    (df['center_lon'] >= la_bbox['west']) &
    (df['center_lon'] <= la_bbox['east']) &
    (df['center_lat'] >= la_bbox['south']) &
    (df['center_lat'] <= la_bbox['north'])
]

print(f"Found {len(filtered_df)} items over Los Angeles")
print(f"Date range: {filtered_df['start_datetime'].min()} to {filtered_df['start_datetime'].max()}")


from tabulate import tabulate

print("\nFiltered items:")
print(tabulate(
    filtered_df[['stac_id', 'datetime', 'center_lon', 'center_lat', 'instrument_mode']],
    headers='keys',
    tablefmt='grid',
    showindex=False
))
# Save to new CSV
filtered_df.to_csv(f"{filename}_filtered.csv", index=False)
print("\nSaved to capella_la_filtered.csv")

Found 8 items over Los Angeles
Date range: 2024-12-11 09:47:26.904082+00:00 to 2025-09-02 02:11:14.960155+00:00

Filtered items:
+-----------------------------------------------------+----------------------------------+--------------+--------------+-------------------+
| stac_id                                             | datetime                         |   center_lon |   center_lat | instrument_mode   |
+=====================================================+==================================+==============+==============+===================+
| CAPELLA_C13_SP_SLC_HH_20250902021114_20250902021153 | 2025-09-02 02:11:34.236420+00:00 |     -118.24  |      34.0735 | spotlight         |
+-----------------------------------------------------+----------------------------------+--------------+--------------+-------------------+
| CAPELLA_C13_SP_GEO_HH_20250902021114_20250902021153 | 2025-09-02 02:11:34.236661+00:00 |     -118.24  |      34.0732 | spotlight         |
+------------------------